In [1]:
import pandas as pd
import os


def read_powcod_sql():
    import sqlite3
    con = sqlite3.connect('cod2205.sq')
    df = pd.read_sql_query('SELECT * FROM id',con)
    con2 = sqlite3.connect('cod2205.sq.info')
    df2 = pd.read_sql_query('SELECT * FROM info',con2)

    df.id = pd.to_numeric(df.id)
    df2.id = pd.to_numeric(df2.id)

    combined = df.merge(df2, left_on="id", right_on="id")
    return combined
if os.path.isfile("combined.pkl"):
    combined = pd.read_pickle("combined.pkl")
else:
    combined = read_powcod_sql()
    print(combined.columns)
    combined = combined[['name', 'mineralname', 'chemical_formula',"spacegroup_x", "a", "b", "c", "alpha", "beta",
                    "gamma", "volume", "h", "k", "l", "nd", "dvalue",
                    "intensita", "type", "id"]]
    combined.to_pickle("combined.pkl")

Index(['id', 'name', 'mineralname', 'chemical_formula', 'spacegroup_x',
       'quality', 'rir_x', 'nrec', 'natoms_x', 'nreflections_x', 'nd',
       'dvalue', 'intensita', 'n', 'authors', 'journal', 'journal_year',
       'journal_volume', 'journal_issue', 'page_start', 'page_end', 'color',
       'crystal_density', 'z', 'spacegroup_y', 'type', 'volume', 'density',
       'mu(CuKa)', 'natoms_y', 'nreflections_y', 'a', 'b', 'c', 'alpha',
       'beta', 'gamma', 'rir_y', 'h', 'k', 'l', 'mul'],
      dtype='object')


In [ ]:
from pymatgen.symmetry import groups
import numpy as np

symbol_to_num = {}
for i, s in enumerate(np.unique(combined.spacegroup_x.to_numpy())):
    try:
        g = groups.SpaceGroup(s)
        symbol_to_num[s] = g.int_number
    except:
        symbol_to_num[s] = -1

def label_SGs(sg):
    return symbol_to_num[sg]

combined["sg_number"] = combined.spacegroup_x.apply(label_SGs)
combined = combined[combined.sg_number > 0]

In [52]:
combined = combined[['chemical_formula','spacegroup_x','sg_number']]

In [54]:
combined.to_csv("PowCOD.csv")

In [5]:
combined['chemical_formula'].isna()

0         False
1         False
2         False
3         False
4         False
          ...  
458155    False
458156    False
458157    False
458158    False
458159    False
Name: chemical_formula, Length: 457069, dtype: bool

In [12]:
combined[combined['chemical_formula'] != ""][['chemical_formula','sg_number']].to_csv("PowCOD.csv")